In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hjp2001610206/toumanfen/train_raw.txt
/kaggle/input/datasets/hjp2001610206/toumanfen/stopwords.txt
/kaggle/input/datasets/hjp2001610206/toumanfen/dev_raw.txt
/kaggle/input/datasets/hjp2001610206/toumanfen/class.txt
/kaggle/input/datasets/hjp2001610206/toumanfen/test_raw.txt


# BERT Model

In [2]:
pip install modelscope

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 71.3 MB/s eta 0:00:00:00:010:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip show modelscope

Name: modelscope
Version: 1.37.1
Summary: ModelScope: bring the notion of Model-as-a-Service to life.
Home-page: https://github.com/modelscope/modelscope
Author: ModelScope team
Author-email: contact@modelscope.cn
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, packaging, requests, setuptools, tqdm, urllib3
Required-by: 


In [4]:
from modelscope import snapshot_download

# 指定你想要保存模型的文件夹路径
my_local_dir = '/kaggle/working/model/' 

model_dir = snapshot_download('hfl/chinese-macbert-base',
                              cache_dir=my_local_dir,
                              ignore_patterns=['*.h5', '*.msgpack']  # 忽略这些格式的文件
                             )

print(f"模型已下载到: {model_dir}")

2026-06-20 14:28:08,742 - modelscope - INFO - Got 9 files, start to download ...


Processing 9 items:   0%|          | 0.00/9.00 [00:00<?, ?it/s]

2026-06-20 14:28:46,285 - modelscope - INFO - Finish downloading 9 files for repo 'hfl/chinese-macbert-base'


模型已下载到: /kaggle/working/model/hfl/chinese-macbert-base


In [5]:
BRET_PATH = model_dir

# Data Process

In [6]:
def data_process(input_txt_path, out_txt_path):
    """
    数据处理,(去除空白符号，重复值)
    :param input_txt_path: 输入txt文件路径
    :param out_txt_path: 输出txt文件路径
    :return:
    """
    lines = set()
    with open(input_txt_path, "r", encoding="utf-8") as f:
        with open(out_txt_path, "w", encoding="utf-8") as fw:
            # 逐行处理，去除空白符号，重复值
            for line in f:
                line = line.strip()
                # 空行跳过
                if line:
                    if line not in lines:
                        lines.add(line)
                        fw.write(line + "\n")

In [7]:
import os
os.makedirs("/kaggle/working/data/", exist_ok=True)

TRAIN_DATA_PATH = "/kaggle/working/data/train.txt"
data_process("/kaggle/input/datasets/hjp2001610206/toumanfen/train_raw.txt",TRAIN_DATA_PATH)

TEST_DATA_PATH = "/kaggle/working/data/test.txt"
data_process("/kaggle/input/datasets/hjp2001610206/toumanfen/test_raw.txt",TEST_DATA_PATH)

DEV_DATA_PATH = "/kaggle/working/data/dev.txt"
data_process("/kaggle/input/datasets/hjp2001610206/toumanfen/dev_raw.txt",DEV_DATA_PATH)


# train_BERT

In [8]:
BRET_PATH

'/kaggle/working/model/hfl/chinese-macbert-base'

In [9]:
import torch
from tqdm import tqdm
from transformers import BertTokenizer, BertConfig, BertModel

In [10]:
BERT_TOKENIZER = BertTokenizer.from_pretrained(BRET_PATH)
BERT_CONFIG = BertConfig.from_pretrained(BRET_PATH)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

save_path = '/kaggle/working/model/hfl/save_model/'
os.makedirs(save_path, exist_ok=True)
bert_save_model_path = '/kaggle/working/model/hfl/save_model/bert_model.pt'

In [11]:
def load_data(txt_path):
    """
    加载数据（已经去重）（BERT无需复杂数据处理）
    :param txt_path:数据路径
    :return:list(tuple(text,label))
    """
    # 1.创建列表
    results = []
    # 2.读取文件
    with open(txt_path, 'r', encoding='utf-8') as f:
        # 1.逐行处理
        for line in f:
            # 2.去除空白符号
            line = line.strip()
            # 3.判断是否为空行
            if line:
                # 4.拆分
                text, label = line.split('\t')
                # 5.添加到列表中
                results.append((text, int(label)))

    return results


from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, data_list):
        super().__init__()
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, index):
        text, label = self.data_list[index]
        return text, label

In [12]:
def collate_fn(batch):
    # 1.获取数据tuple,tuple
    texts, labels = zip(*batch)
    # 2.
    output = BERT_TOKENIZER(
        list(texts),
        max_length=32,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )
    # print(output)
    # 输出：{'input_ids':token编码,
    #       'token_type_ids':句子编码（上下句子）,
    #       'attention_mask':mask}
    # 3.获取input_ids，attention_mask
    input_ids = output['input_ids'].long()
    attention_mask = output['attention_mask'].long()
    # 4.label
    labels = torch.tensor(labels).long()
    return input_ids, attention_mask, labels


def build_dataloader():
    # 1.加载数据
    train_data = load_data(TRAIN_DATA_PATH)
    dev_data = load_data(DEV_DATA_PATH)
    test_data = load_data(TEST_DATA_PATH)
    # 2.创建数据集
    train_dataset = MyDataset(train_data)
    dev_dataset = MyDataset(dev_data)
    test_dataset = MyDataset(test_data)
    batch_size = 64
    # 3.创建数据加载器
    train_dataloader = torch.utils.data.DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )
    dev_dataloader = torch.utils.data.DataLoader(
        dataset=dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )
    test_dataloader = torch.utils.data.DataLoader(
        dataset=test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )
    return train_dataloader, dev_dataloader, test_dataloader

In [13]:

class MyBertModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # 1.创建bert模型
        self.bert = BertModel.from_pretrained(BRET_PATH)
        # 2.创建全连接层
        self.linear1 = torch.nn.Linear(BERT_CONFIG.hidden_size, 10)
        # 3.冻结bert
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        # 1.获取bert的输出
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # 2.获取pooler_output
        pooler_output = output.pooler_output

        # 3.全连接层
        logits = self.linear1(pooler_output)
        # 4.返回
        return logits

In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, \
    confusion_matrix


def get_score(y_true, y_pred, print_result=False):
    # 准确率 = 预测正确的样本数 / 总样本数
    acc = accuracy_score(y_true, y_pred)

    # macro: 宏平均，计算各个类比指标，再平均，每个类别权重一样
    # weighted: 加权平均，计算各个类比指标，再平均，每个类别权重按照样本数量
    # micro: 微平均，计算各个类别的TP、FP、FN，再汇总为总的TP、FP、FN，按照公式来计算各个指标
    # 精确率 = 真正例的样本数 / 所有预测为正的样本数 = TP/(TP+FP)
    prec = precision_score(y_true, y_pred, average='macro')

    # 召回率 = 真正例的样本数 / 所有正样本数 = TP/(TP+FN)
    rec = recall_score(y_true, y_pred, average='macro')

    # f1 = 2*P*R/(P+R)
    f1 = f1_score(y_true, y_pred, average='macro')

    if print_result:
        print(f"准确率: {acc}")
        print(f"精确率: {prec}")
        print(f"召回率: {rec}")
        print(f"F1-score: {f1}")
        print(f"分类评估报告: \n{classification_report(y_true, y_pred)}")
        print(f"混淆矩阵: \n{confusion_matrix(y_true, y_pred)}")

    return acc, prec, rec, f1

In [16]:

def train_one_epoch(model, dataloader, loss_fn, optimizer):
    # 1.训练模式
    model.train()
    # 2.记录损失等
    total_loss = 0.0
    total_samples = 0
    total_preds = []
    total_labels = []
    for batch in tqdm(dataloader):
    # for batch in dataloader:
        # 1.迁移到设备
        input_ids, attention_mask, labels = [x.to(DEVICE) for x in batch]
        # 2.前向传播
        logits = model(input_ids, attention_mask)
        # 3.计算损失
        loss = loss_fn(logits, labels)
        # 4.梯度清零
        optimizer.zero_grad()
        # 5.反向传播
        loss.backward()
        # 6.优化器更新
        optimizer.step()
        # 7.记录损失
        total_loss += loss.item() * input_ids.size(0)
        total_samples += input_ids.size(0)
        total_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        total_labels.extend(labels.cpu().numpy())
    # 3.计算损失
    loss = total_loss / total_samples
    score = get_score(total_labels, total_preds)
    acc = score[0]
    f1 = score[3]
    return loss, acc, f1


@torch.no_grad()
def evaluate(model, dev_dataloader, loss_fn):
    # 1.训练模式
    model.eval()
    # 2.记录损失等
    total_loss = 0.0
    total_samples = 0
    total_preds = []
    total_labels = []
    for batch in tqdm(dev_dataloader):
    # for batch in dev_dataloader:
        # 1.迁移到设备
        input_ids, attention_mask, labels = [x.to(DEVICE) for x in batch]
        # 2.前向传播
        logits = model(input_ids, attention_mask)
        # 3.计算损失
        loss = loss_fn(logits, labels)
        # # 4.梯度清零
        # optimizer.zero_grad()
        # # 5.反向传播
        # loss.backward()
        # # 6.优化器更新
        # optimizer.step()
        # 7.记录损失
        total_loss += loss.item() * input_ids.size(0)
        total_samples += input_ids.size(0)
        total_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        total_labels.extend(labels.cpu().numpy())
    # 3.计算损失
    loss = total_loss / total_samples
    score = get_score(total_labels, total_preds)
    acc = score[0]
    f1 = score[3]
    return loss, acc, f1


def train():
    # 1.数据加载器
    train_dataloader, dev_dataloader, _ = build_dataloader()
    # 2.创建模型
    model = MyBertModel().to(DEVICE)
    # 3.创建损失函数
    loss_fn = torch.nn.CrossEntropyLoss()
    # 4.创建优化器
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    # 5.初始化训练列表
    train_loss_list = []
    train_acc_list = []
    train_f1_list = []
    dev_loss_list = []
    dev_acc_list = []
    dev_f1_list = []
    best_f1 = 0.0
    # 6.训练
    epochs = 2
    for epoch in range(epochs):
        # 1.训练
        train_loss, train_acc, train_f1 = train_one_epoch(model, train_dataloader, loss_fn, optimizer)
        # 2.验证
        dev_loss, dev_acc, dev_f1 = evaluate(model, dev_dataloader, loss_fn)
        # 3.保存模型
        if dev_f1 > best_f1:
            best_f1 = dev_f1
            torch.save(model.state_dict(), bert_save_model_path)
        # 4.记录
        train_loss_list.append(train_loss)
        train_acc_list.append(train_acc)
        train_f1_list.append(train_f1)
        dev_loss_list.append(dev_loss)
        dev_acc_list.append(dev_acc)
        dev_f1_list.append(dev_f1)
        print(f"epoch: {epoch + 1}/{epochs}")
        print(f"train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, train_f1: {train_f1:.4f}")
        print(f"dev_loss: {dev_loss:.4f}, dev_acc: {dev_acc:.4f}, dev_f1: {dev_f1:.4f}")
    return {
        "train_loss": train_loss_list,
        "train_acc": train_acc_list,
        "train_f1": train_f1_list,
        "dev_loss": dev_loss_list,
        "dev_acc": dev_acc_list,
        "dev_f1": dev_f1_list
    }

In [ ]:
train()